# 第一个 CUDA Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-08-cuda-kernel.ipynb)

亲手写并行 GPU 代码。用 Numba 和纯 CUDA C++ 两种方式实现千万级向量加法，理解 Grid / Block / Thread / Warp 怎么映射到 T4 的 40 个 SM，并用 CUDA Event 测出 kernel 真实耗时。

本 notebook 配合教程章节 [第一个 CUDA Kernel](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

后面 HAMi 为什么要“切算力”（`nvidia.com/gpucores`）、为什么 MPS 能让两个进程的 kernel 交替占用 SM、为什么 time-slicing 会带来抖动，全都建立在“kernel 是怎么被分发到 SM 上的”这件事之上。本 Lab 用一个最小但规模足够大的 kernel，让你看清 Grid → Block → Thread 的映射，把 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 里“SM 在执行 kernel”这句话落到实处。

本 Lab 给出两个版本：**Numba**（Python 写 CUDA kernel，运行时 JIT，适合快速迭代）和**纯 CUDA C++**（用 [Lab 07](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/) 的 `nvcc`，这是生产环境的写法）。两者编译出的 PTX 落到同一套 SM 上。

## Grid / Block / Warp 心智模型

![Grid / Block / Warp 心智模型：Grid 由多个 Block 组成，Block 被分发到 SM，SM 以 32 线程的 Warp 为调度单位](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/grid-block-warp.svg)



| 概念 | 是什么 | T4 上的尺度 |
| :--- | :--- | :--- |
| Thread | 最小执行单元 | — |
| Warp | 32 个线程，SM 的实际调度单位 | 32 |
| Block | 一组协作线程（可共享 shared memory），整体派到一个 SM | 上限 1024 线程 |
| Grid | 一次 kernel launch 的全部 block | 由你指定 |
| SM | T4 的流式多处理器，block 被派给它 | 40 个 SM，每个最多同时驻留 16 个 block / 1024 个线程 |



用代码把这些数字从硬件里读出来：

In [ ]:
from numba import cuda
dev = cuda.get_current_device()
print("name            :", dev.name.decode() if isinstance(dev.name, bytes) else dev.name)
print("compute cap     :", dev.compute_capability)
print("SM count        :", dev.MULTIPROCESSOR_COUNT)
print("warp size       :", dev.WARP_SIZE)
print("max threads/blk :", dev.MAX_THREADS_PER_BLOCK)

实测输出：

```text
name            : Tesla T4
compute cap     : ComputeCapability(major=7, minor=5)
SM count        : 40
warp size       : 32
max threads/blk : 1024
```

这几个数就是上表 T4 那一列：40 个 SM、每个 warp 32 个线程、一个 block 最多 1024 个线程。后面选 `threads_per_block=256`，就是在 1024 的上限内取一个 32 的整数倍。

## 第 1 步：用 Numba 写向量加法

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def add_kernel(a, b, c):
    i = cuda.grid(1)          # 全局线程索引
    n = c.size
    if i < n:                 # 越界保护
        c[i] = a[i] + b[i]

`cuda.grid(1)` 帮你算出“我是第几个线程”，等价于 [Lab 07](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/) 里的 `blockIdx.x * blockDim.x + threadIdx.x`。

## 第 2 步：准备数据并启动

In [ ]:
N = 10_000_000
a = np.ones(N, dtype=np.float32)
b = np.full(N, 2.0, dtype=np.float32)
c = np.zeros(N, dtype=np.float32)

da = cuda.to_device(a); db = cuda.to_device(b); dc = cuda.to_device(c)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

add_kernel[blocks_per_grid, threads_per_block](da, db, dc)
cuda.synchronize()

result = dc.copy_to_host()
print("first 5:", result[:5], "expected all 3.0")
print("check ok:", np.allclose(result, 3.0))

实测输出：

```text
first 5: [3. 3. 3. 3. 3.] expected all 3.0
check ok: True
```

1 + 2 = 3，一千万个元素全部算对。顺便算一下：`blocks_per_grid` = 39063，而 T4 只有 40 个 SM，每个 SM 最多同时驻留 16 个 block，所以这 39063 个 block 是**分批**派发的：硬件调度器把跑完的 block 换下、新 block 换上，对你完全透明。这就是 CUDA“自动并行、自动扩展”的来源：同一份代码在 40 个 SM 的 T4 和 132 个 SM 的 H100 上都能跑，只是批次数不同。

> 注意 [Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 学到的点：`to_device` 把数据从 CPU 搬到 GPU，`copy_to_host` 搬回来，这就是 [Lab 02](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 说的“尽量少来回搬”。本例只在首尾各搬一次。

## 第 3 步：观察它让 GPU-Util 冲高

In [ ]:
import subprocess, time
def util():
    return subprocess.check_output(["nvidia-smi","--query-gpu=utilization.gpu","--format=csv,noheader,nounits"]).decode().strip().splitlines()[0]

# 连跑 2000 次 kernel（每次约 0.5 ms，合计约 1 秒，足够跨过一个采样窗口）
t0 = time.time()
for _ in range(2000):
    add_kernel[blocks_per_grid, threads_per_block](da, db, dc)
cuda.synchronize()
print("2000 launches took %.2f s" % (time.time() - t0))
print("util right after :", util(), "%")

实测输出：

```text
2000 launches took 0.93 s
util right after : 100 %
```

2000 次用了 0.93 秒，平均每次约 0.47 ms；紧接着采样，`GPU-Util` 是 100%。

但 100% 不代表 GPU 的算力被用满了。向量加法是**内存带宽受限**的：每个元素读 8 字节、写 4 字节、只做 1 次加法。N = 10⁷ 时每次 kernel 搬 120 MB，按 T4 的 320 GB/s 算约 0.4 ms，和实测的 0.47 ms 很接近：时间几乎全花在搬数据上，计算单元大部分时间在等。这正是 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 说的“100% ≠ 饱和”。

## 第 4 步：纯 CUDA C++ 版本，并用 CUDA Event 计时

In [ ]:
%%writefile vector_add.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \
    fprintf(stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)

__global__ void add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int N = 10'000'000;
    size_t bytes = N * sizeof(float);
    float *ha = (float*)malloc(bytes), *hb = (float*)malloc(bytes), *hc = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { ha[i] = 1.0f; hb[i] = 2.0f; }

    float *da, *db, *dc;
    CHECK(cudaMalloc(&da, bytes)); CHECK(cudaMalloc(&db, bytes)); CHECK(cudaMalloc(&dc, bytes));
    CHECK(cudaMemcpy(da, ha, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db, hb, bytes, cudaMemcpyHostToDevice));

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start)); CHECK(cudaEventCreate(&stop));
    add<<<blocks, threads>>>(da, db, dc, N);       // 预热，让驱动加载模块
    CHECK(cudaEventRecord(start));
    for (int r = 0; r < 100; r++) add<<<blocks, threads>>>(da, db, dc, N);
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    float ms = 0; CHECK(cudaEventElapsedTime(&ms, start, stop));

    CHECK(cudaMemcpy(hc, dc, bytes, cudaMemcpyDeviceToHost));
    int bad = 0; for (int i = 0; i < N; i++) if (hc[i] != 3.0f) bad++;

    printf("grid=%d blocks x %d threads, N=%d\n", blocks, threads, N);
    printf("avg kernel time : %.3f ms\n", ms / 100);
    printf("effective bw    : %.1f GB/s\n", 3.0 * bytes / (ms / 100 / 1000) / 1e9);
    printf("check           : %s\n", bad == 0 ? "ok" : "FAILED");

    cudaFree(da); cudaFree(db); cudaFree(dc); free(ha); free(hb); free(hc);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o vector_add vector_add.cu && ./vector_add

实测输出（耗时和带宽每次略有浮动）：

```text
grid=39063 blocks x 256 threads, N=10000000
avg kernel time : 0.468 ms
effective bw    : 256.5 GB/s
check           : ok
```

C++ 版每次 kernel 0.468 ms，和第 3 步 Numba 版的 0.47 ms 几乎一样：两者最终跑的是同一种 GPU 指令，瓶颈都在显存带宽。

对照看三件事：

- `add<<<blocks, threads>>>` 就是 Numba 里 `add_kernel[blocks_per_grid, threads_per_block]` 藏掉的启动语法；
- `cudaMalloc` / `cudaMemcpy` 对应 `cuda.to_device` / `copy_to_host`，现在你看到它们是显式的 CUDA Runtime API 调用，这也是 HAMi-core 用 `LD_PRELOAD` 拦截的那一层；
- **CUDA Event** 是在 GPU 时间线上打点的正确计时方式。用 CPU 的 `time.time()` 夹住一个异步 launch 只能测到“提交耗时”，不是执行耗时。

有效带宽能到 T4 理论值 320 GB/s 的 80% 左右，说明这个 kernel 已经把显存带宽“跑满”了，再加 SM 也不会更快。这是区分“计算受限”和“带宽受限”的第一个亲身例子。

## 第 5 步：故意越界

把 C++ 版本里的 `if (i < n)` 删掉再跑一次，然后用 Toolkit 自带的内存检查器：

In [ ]:
!sed 's/if (i < n) //' vector_add.cu > oob.cu && nvcc -arch=sm_75 -o oob oob.cu && compute-sanitizer --tool memcheck ./oob 2>&1 | head -15

实测输出（前 15 行，地址每次不同）：

```text
========= COMPUTE-SANITIZER
========= Invalid __global__ read of size 4 bytes
=========     at add(const float *, const float *, float *, int)+0x70
=========     by thread (160,0,0) in block (39062,0,0)
=========     Access to 0x7a83fe625a80 is out of bounds
=========     and is 129 bytes after the nearest allocation at 0x7a83fc000000 of size 40,000,000 bytes
...
```

逐行解读：

- `Invalid __global__ read of size 4 bytes`：某个线程读了 4 字节（一个 float）不该读的显存。是 read 而不是 write，因为 `c[i] = a[i] + b[i]` 要先读 `a[i]`，越界在读的时候就被抓到了；
- `thread (160,0,0) in block (39062,0,0)`：出错的是最后一个 block（编号 39062）里的线程。39063 × 256 = 10000128 > N，这个 block 里编号 128 以后的线程都在数组外；
- `of size 40,000,000 bytes`：越界的那块显存正是 `cudaMalloc` 分配的 1000 万个 float（40 MB），访问地址落在它末尾之后。没有 `compute-sanitizer` 的话这种 bug 可能静默地跑过去，直到某天踩坏别的数据。

把线程编号和数组下标对齐来看，越界从哪来就很清楚了：

![N 为一千万时需要 39063 个 block，最后一个 block 的线程 128 到 255 落在数组末尾之后，sanitizer 抓到的 thread 160 对应下标 10000032](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/out-of-bounds-threads.svg)

## 验证

完成后你应该能：

- 解释 `cuda.grid(1)` 等价于哪两个内建变量；
- 说出 Warp 的大小（32）和它为什么重要；
- 算出给定 N 和 `threads_per_block` 时需要多少个 block，并解释为什么必须有 `if i < n`；
- 用 CUDA Event 而不是 CPU 计时器测 kernel 耗时；
- 解释为什么向量加法是带宽受限、它和 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) “100% ≠ 饱和”的关系。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `numba.cuda` 报 `CudaSupportError` / 无设备 | runtime 是 CPU | 切到 T4 |
| Numba 报 `NvvmSupportError` / 找不到 libnvvm | Numba 与系统 CUDA 版本不匹配 | `pip install -U numba nvidia-cuda-nvcc-cu12 nvidia-cuda-runtime-cu12`，重启 runtime |
| 结果含 0 或乱数 | 没做越界保护 / 没同步 | 加 `if i < n`、`synchronize()` |
| `blocks_per_grid` 算错 | 向上取整写错 | 用 `(N + tpb - 1)//tpb` |
| `nvcc` 报 `Unsupported gpu architecture` | Toolkit 版本与 `-arch` 不匹配 | 改用 `-arch=native` |
| `compute-sanitizer: command not found` | 不在 PATH | `/usr/local/cuda/bin/compute-sanitizer` |
| C++ 版 `avg kernel time` 明显高于 1 ms | 没加 `-O2`，或 GPU 被其它 cell 占用 | 加 `-O2`；停掉 Lab 05 的后台线程 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。